# Library Database Lab — Phân tích từng bước

Dữ liệu mẫu GIẢ LẬP seed 42. Chạy từ trên xuống. Tìm hiểu sự khác nhau giữa đầu sách, bản sao và lượt mượn trước khi đếm. Ngày tham chiếu mẫu: 2025-12-31.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "library_lab").is_dir(), "Mở notebook từ repo hoặc notebooks"
sys.path.insert(0, str(ROOT))
from library_lab.db import connect
from library_lab.init_db import initialize
DB = ROOT / "database/library.db"  # Thay đường dẫn khi dùng database khác
AS_OF = "2025-12-31"              # Thay ngày khi dùng dữ liệu thật
if not DB.exists():
    initialize(DB, sample=True)
sns.set_theme(style="whitegrid", font="DejaVu Sans")

## 1. Đọc bốn bảng

Một dòng books là đầu sách; book_copies là một cuốn; loans là một lượt mượn. members là độc giả.

In [ ]:
with connect(DB, readonly=True) as conn:
    books = pd.read_sql_query("SELECT * FROM books", conn)
    copies = pd.read_sql_query("SELECT * FROM book_copies", conn)
    members = pd.read_sql_query("SELECT * FROM members", conn)
    loans = pd.read_sql_query("SELECT * FROM loans", conn)
print({"books": len(books), "copies": len(copies), "members": len(members), "loans": len(loans)})
display(books.head(), loans.head())

## 2. Kiểm tra chất lượng

returned_on có thể NULL vì chưa trả. Các cột ngày khác không được thiếu; khóa ngoại phải tồn tại.

In [ ]:
display(loans.isna().sum().to_frame("missing"))
assert copies["copy_code"].is_unique
assert loans["loan_id"].is_unique
assert copies["book_id"].isin(books["book_id"]).all()
assert loans["copy_id"].isin(copies["copy_id"]).all()
assert loans["member_id"].isin(members["member_id"]).all()
with connect(DB, readonly=True) as conn:
    assert not conn.execute("PRAGMA foreign_key_check").fetchall()

## 3. JOIN để hiểu lịch sử mượn

JOIN loans → book_copies → books → members. Tất cả giá trị nhập từ người dùng phải dùng tham số SQL.

In [ ]:
query = """SELECT l.loan_id,b.title,c.copy_code,m.name,l.borrowed_on,l.due_on,l.returned_on
FROM loans l JOIN book_copies c ON c.copy_id=l.copy_id
JOIN books b ON b.book_id=c.book_id JOIN members m ON m.member_id=l.member_id
ORDER BY l.loan_id"""
with connect(DB, readonly=True) as conn:
    history = pd.read_sql_query(query, conn)
history.head()

## 4. Quá hạn và còn sẵn

Hạn trả bằng ngày tham chiếu chưa quá hạn. Ngày tham chiếu chỉ dùng phân loại trạng thái hiện có, không tái dựng lịch sử quá khứ.

In [ ]:
from library_lab.service import active_loans, available_copies
print("Đang mượn:", len(active_loans(DB, as_of=AS_OF)))
print("Đang quá hạn:", len(active_loans(DB, overdue_only=True, as_of=AS_OF)))
print("Bản sao còn sẵn:", len(available_copies(DB)))
display(pd.DataFrame(active_loans(DB, overdue_only=True, as_of=AS_OF)))

## 5. Đối chiếu doanh số mượn bằng SQL và pandas

Đếm lượt, không đếm người duy nhất. Tháng không phát sinh lượt cần được phân biệt với dữ liệu bị thiếu.

In [ ]:
with connect(DB, readonly=True) as conn:
    sql_monthly = pd.read_sql_query("SELECT strftime('%Y-%m',borrowed_on) AS month,COUNT(*) AS loan_count FROM loans GROUP BY month ORDER BY month", conn)
pandas_monthly = loans.assign(month=pd.to_datetime(loans["borrowed_on"]).dt.strftime("%Y-%m")).groupby("month", as_index=False).size().rename(columns={"size": "loan_count"})
pd.testing.assert_frame_equal(sql_monthly, pandas_monthly)
assert sql_monthly["loan_count"].sum() == len(loans)
sql_monthly

## 6. Vẽ Seaborn

Trục phải có tên và đơn vị. Nếu database rỗng, không suy ra kết luận từ biểu đồ trống.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if not sql_monthly.empty:
    sns.lineplot(data=sql_monthly, x="month", y="loan_count", marker="o", ax=ax)
else:
    ax.text(.5,.5,"Chưa có lượt mượn",ha="center",transform=ax.transAxes)
ax.set(title="Lượt mượn theo tháng", xlabel="Tháng", ylabel="Số lượt")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 7. Phân tích thời gian của lượt đã trả

Không điền 0 cho lượt chưa trả; đây là dữ liệu chưa quan sát đầy đủ.

In [ ]:
from library_lab.analyze import load_tables
tables = load_tables(DB)
display(tables["returned"]["duration_days"].describe())
print("Độc giả từng mượn:", loans["member_id"].nunique())

## 8. Xuất bốn biểu đồ và báo cáo

Phần xuất kết quả dùng cùng hàm với terminal; không sửa dữ liệu trong database.

In [ ]:
from library_lab.analyze import analyze
analyze(DB, ROOT / "outputs", AS_OF)
print((ROOT / "outputs/report.md").read_text(encoding="utf-8"))

## 9. Tự viết kết luận

- Ba phát hiện kèm số liệu:
- Hai câu hỏi cần dữ liệu thật:
- Giới hạn của số lượt mượn và thời gian mượn:

Làm bài tập trong docs/exercises.md trước khi đọc lời giải. Dữ liệu mẫu không phản ánh nhu cầu đọc thực tế.